# Airfoil Preprocessing and Latent-Space Analysis

This notebook builds a complete preprocessing workflow for airfoil data: dataset loading, geometric normalization, CST/Kulfan parameterization checks, aerodynamic augmentation, clustering, and VAE-based latent analysis.

Use it as a step-by-step pipeline: choose one dataset-loading option, run preprocessing/benchmark sections, then proceed to clustering and latent-space diagnostics.

## Importing Modules


In [1]:
import gc
import io
import os
import random
import requests
import shutil
import sys
import time
from pathlib import Path

import aerosandbox as asb
import aerosandbox.numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from aerosandbox import Airfoil
from aerosandbox.geometry.airfoil.airfoil_families import get_kulfan_coordinates, get_kulfan_parameters
from datasets import load_dataset
from matplotlib.cm import ScalarMappable
from matplotlib.colors import to_rgba
from matplotlib.patches import Ellipse, RegularPolygon
from minisom import MiniSom
from omegaconf import OmegaConf
from scipy.stats import multivariate_normal
from sklearn.cluster import KMeans, SpectralBiclustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler, StandardScaler

from generative_inverse_wing_design.DataGenerator.utils import (
    analyze_latent_distributions_l2,
    assess_cluster_quality_simple,
    assign_hex_clusters,
    calculate_point_grid_iou_polygon,
    compute_p_x_given_k,
    cst_airfoil,
    fit_cst_to_airfoil,
    is_trailing_edge_closed,
    load_all_dat_files,
    map_dataset_to_latent_space,
    normalize_airfoil,
    sample_from_latent_grid,
    visualize_airfoil_reconstruction,
    visualize_latent_grid_sampling,
    visualize_latent_space,
    visualize_outlier_airfoils,
)
from generative_inverse_wing_design.models.VAE.preprocessing_VAE import VAE
from generative_inverse_wing_design.models.utils.training_utils import train_preprocessing_vae_with_hydra


## Dataset selection
Choose one dataset-loading block below and execute only that block before continuing.

This notebook supports multiple sources (AeroSandbox database, IDEALLab arrays, and HuggingFace dataset format).

### Option A — Load AeroSandbox `.dat` airfoil database
This path reads local `.dat` files and builds an object-array of variable-length coordinate sets plus matching names.

In [6]:
# Load all airfoils using relative path
dat_folder_path = Path(r"data/external/airfoil_database")

airfoils_asb, airfoils_names_asb = load_all_dat_files(dat_folder_path)

print(f"\nLoaded {len(airfoils_asb)} airfoils successfully")
print(f"Airfoil names array shape: {airfoils_names_asb.shape}")
print(f"Airfoils array shape: {airfoils_asb.shape}")

if len(airfoils_asb) > 0:
    # Show statistics about point counts
    point_counts = np.array([coords.shape[0] for coords in airfoils_asb])
    print(f"\nPoint count statistics:")
    print(f"  Min points: {point_counts.min()}")
    print(f"  Max points: {point_counts.max()}")
    print(f"  Mean points: {point_counts.mean():.1f}")
    print(f"  Std points: {point_counts.std():.1f}")

X_airfoils = airfoils_asb


Loaded 0 airfoils successfully
Airfoil names array shape: (0,)
Airfoils array shape: (0,)


### Option B — Load IDEALLab textbook dataset
This block downloads a pre-interpolated NumPy array (`airfoil_interp_uniform.npy`) where all airfoils share the same point count.

In [ ]:
# Download the airfoil dataset from GitHub
url = "https://github.com/IDEALLab/ML4ME_Textbook/raw/main/part1/airfoil_interp_uniform.npy"

# Download the data file
response = requests.get(url)
response.raise_for_status()  # raise an error if download failed

# Load into numpy array
X_airfoils = np.load(io.BytesIO(response.content))
print(f"Downloaded {X_airfoils.shape[0]} airfoils with {X_airfoils.shape[1]} points each.")


### Option C — Load IDEALLab/EngiBench dataset
This block merges train/val/test splits into one coordinate array and sets `X_airfoils` as the common downstream input.

In [ ]:
design = "optimal_design"  # or "initial_design"

dataset = load_dataset("IDEALLab/airfoil_v0")
print(dataset["train"].features)
num_airfoils = len(dataset["train"][design]) + len(dataset["val"][design]) + len(dataset["test"][design])
print(f"Dataset contains {num_airfoils} airfoils total.")

full_data = []
for key in ["train", "val", "test"]:
    for i in range(len(dataset[key]["initial_design"])):
            full_data.append(np.array(dataset[key]["initial_design"][i]["coords"]).T)
full_data = np.array(full_data)
print(f"Loaded {full_data.shape[0]} airfoils with {full_data.shape[1]} points each.")
X_airfoils = full_data

## Normalize geometry to a common frame
Airfoils are normalized to unit chord and aligned so comparisons and CST fitting are consistent across sources.

Assumed ordering: trailing edge (upper) → leading edge → trailing edge (lower).

In [ ]:
# Geometry normalization helper now lives in DataGenerator.utils.
# Quick smoke-check on one sample from the selected dataset.
_normalized_example = normalize_airfoil(X_airfoils[0])
print(f"Normalized example shape: {_normalized_example.shape}")

Check for normalized airfoils to have a closed trailing edge, as we only work with closed trailing edge airfoils

In [ ]:
open_te_indices = []
for idx, airfoil in enumerate(X_airfoils):
    if airfoil is not None and len(airfoil) > 0:
        if not is_trailing_edge_closed(airfoil):
            open_te_indices.append(idx)

print(f"Found {len(open_te_indices)} airfoils with open trailing edges: {open_te_indices}")

## CST/Kulfan parameter extraction and reconstruction
This section fits Kulfan/CST coefficients to an example airfoil and reconstructs coordinates to inspect parameterization quality.

In [ ]:
airfoil_id =505
dim_per_side = 3  # Number of weights per side for ASB Kulfan
n_points_per_side = 96 # halve of the total number of points of IDEALLAB airfoils

results = get_kulfan_parameters(X_airfoils[airfoil_id], n_weights_per_side=dim_per_side, use_leading_edge_modification=True)
lw = results["lower_weights"]
uw = results["upper_weights"]
tw = results["TE_thickness"]
le = results["leading_edge_weight"]

print("Lower weights:", lw)
print("Upper weights:", uw)
print("Trailing edge thickness:", tw)
print("Leading edge weight:", le)

#overwrite to close trailing edge
tw = 0

coords_reconstructed = get_kulfan_coordinates(lower_weights=lw,
                                              upper_weights=uw,
                                              leading_edge_weight=le,
                                              TE_thickness=tw,
                                              n_points_per_side=n_points_per_side)


### Visual comparison and IoU metric
Original and reconstructed point clouds are plotted side-by-side, and a polygon-based IoU score quantifies shape overlap.

In [ ]:
x_original = X_airfoils[airfoil_id][:, 0]
y_original = X_airfoils[airfoil_id][:, 1]

x_reconstructed = coords_reconstructed[:, 0]
y_reconstructed = coords_reconstructed[:, 1]

plt.figure(figsize=(20, 10))
plt.subplot(1, 2, 1)
plt.scatter(x_original, y_original, alpha=0.5, label='Original points', s=10)
plt.scatter(x_reconstructed, y_reconstructed, alpha=0.5, s=10, label='reconstructed', color='red')
plt.axis('equal')
plt.legend()
plt.title('CST Fit Comparison')
plt.grid(True, alpha=0.3)

print("IoU (Polygon Approximation):", calculate_point_grid_iou_polygon(X_airfoils[airfoil_id], coords_reconstructed))

## Custom CST fitting baseline
This section runs a custom least-squares CST fit, then compares it with the AeroSandbox Kulfan parameterization using the same reconstruction and IoU metric.

In [ ]:
# Example usage with your airfoil data
airfoil_points = X_airfoils[airfoil_id]

# Fit CST parameters (shared utility implementation)
Au_fitted, Al_fitted, le_fitted = fit_cst_to_airfoil(airfoil_points, n_cst=dim_per_side)

print(f"Upper surface CST coefficients: {Au_fitted}")
print(f"Lower surface CST coefficients: {Al_fitted}")
print(f"Leading edge weight: {le_fitted}")

In [ ]:
x_fit = np.cosspace(0, 1, n_points_per_side)
yu_reconstructed, yl_reconstructed = cst_airfoil(x_fit, Au_fitted, Al_fitted, le_fitted)

x_reconstructed = np.concatenate([x_fit[::-1], x_fit[1:]])
y_reconstructed = np.concatenate([yu_reconstructed[::-1], yl_reconstructed[1:]])

plt.figure(figsize=(20, 10))
plt.subplot(1, 2, 1)
plt.scatter(x_original, y_original, alpha=0.5, label='Original points', s=10)
plt.scatter(x_reconstructed, y_reconstructed, alpha=0.5, s=10, label='reconstructed', color='red')
plt.axis('equal')
plt.legend()
plt.title('CST Fit Comparison')
plt.grid(True, alpha=0.3)

print("IoU (Polygon Approximation):", calculate_point_grid_iou_polygon(X_airfoils[airfoil_id], np.array([x_reconstructed, y_reconstructed]).T))

## Benchmark custom CST fit vs AeroSandbox Kulfan fit
This section evaluates both parameterization pipelines over the full dataset using a common reconstruction quality metric (IoU).

> Decision criterion: if custom fitting does not materially improve IoU, prefer the more stable/maintainable AeroSandbox implementation.

In [ ]:
# Create a pandas DataFrame with both custom CST and ASB Kulfan parameters for all airfoils

# Initialize lists to store the parameters
# ASB parameters
asb_upper_weights_all = []
asb_lower_weights_all = []
asb_leading_edge_weights = []
asb_iou_values = []

# Custom CST parameters
custom_upper_weights_all = []
custom_lower_weights_all = []
custom_leading_edge_weights = []
custom_iou_values = []

# Process each airfoil in the dataset
print(f"Processing {X_airfoils.shape[0]} airfoils...")
start_time = time.time()

for i in range(X_airfoils.shape[0]):
    if i % 100 == 0:  # Progress indicator
        print(f"Processing airfoil {i}/{X_airfoils.shape[0]}")
    
    # Get ASB Kulfan parameters for current airfoil
    asb_results = get_kulfan_parameters(X_airfoils[i], 
                                       n_weights_per_side=dim_per_side, 
                                       use_leading_edge_modification=True)
    
    # Store ASB results
    asb_upper_weights_all.append(asb_results["upper_weights"])
    asb_lower_weights_all.append(asb_results["lower_weights"])
    asb_leading_edge_weights.append(asb_results["leading_edge_weight"])
    
    # Reconstruct airfoil using ASB parameters and calculate IoU
    asb_coords_reconstructed = get_kulfan_coordinates(
        lower_weights=asb_results["lower_weights"],
        upper_weights=asb_results["upper_weights"],
        leading_edge_weight=asb_results["leading_edge_weight"],
        TE_thickness=0,  # Close trailing edge
        n_points_per_side=n_points_per_side
    )
    asb_iou = calculate_point_grid_iou_polygon(X_airfoils[i], asb_coords_reconstructed)
    asb_iou_values.append(asb_iou)
    
    try:
        Au_fitted, Al_fitted, le_fitted = fit_cst_to_airfoil(X_airfoils[i], n_cst=dim_per_side)
        
        # Store custom results
        custom_upper_weights_all.append(Au_fitted)
        custom_lower_weights_all.append(Al_fitted)
        custom_leading_edge_weights.append(le_fitted)
        
        # Reconstruct airfoil using custom parameters via ASB function as it is the same as the custom one in therms of cst --> coords
        custom_coords_reconstructed = get_kulfan_coordinates(
            lower_weights=Al_fitted,
            upper_weights=Au_fitted,
            leading_edge_weight=le_fitted,
            TE_thickness=0,  # Close trailing edge
            n_points_per_side=n_points_per_side
        )
        custom_iou = calculate_point_grid_iou_polygon(X_airfoils[i], custom_coords_reconstructed)
        custom_iou_values.append(custom_iou)
        
    except Exception as e:
        # Handle cases where custom fitting fails
        print(f"Custom fitting failed for airfoil {i}: {e}")
        custom_upper_weights_all.append([0.0, 0.0, 0.0])
        custom_lower_weights_all.append([0.0, 0.0, 0.0])
        custom_leading_edge_weights.append(0.0)
        custom_iou_values.append(0.0)

end_time = time.time()
print(f"Processing completed in {end_time - start_time:.2f} seconds")

# Create the comprehensive DataFrame - adjusted for dim_per_side
data = {}

# ASB parameters - dynamically create columns based on dim_per_side
for i in range(dim_per_side):
    data[f'asb_upper_{i}'] = [uw[i] if len(uw) > i else 0.0 for uw in asb_upper_weights_all]
    data[f'asb_lower_{i}'] = [lw[i] if len(lw) > i else 0.0 for lw in asb_lower_weights_all]

data['asb_leading_edge_weight'] = asb_leading_edge_weights
data['asb_iou'] = asb_iou_values

# Custom CST parameters - dynamically create columns based on dim_per_side
for i in range(dim_per_side):
    data[f'custom_upper_{i}'] = [uw[i] if len(uw) > i else 0.0 for uw in custom_upper_weights_all]
    data[f'custom_lower_{i}'] = [lw[i] if len(lw) > i else 0.0 for lw in custom_lower_weights_all]

data['custom_leading_edge_weight'] = custom_leading_edge_weights
data['custom_iou'] = custom_iou_values

comparison_df = pd.DataFrame(data)

# Display the DataFrame info and first few rows
print(f"\nDataFrame created with shape: {comparison_df.shape}")
print(f"Number of dimensions per side: {dim_per_side}")
print("\nFirst 10 rows:")
print(comparison_df.head(10))

print("\nDataFrame info:")
print(comparison_df.info())

print("\nBasic statistics:")
print(comparison_df.describe())

# Compare IoU values
print(f"\nASB IoU - Mean: {comparison_df['asb_iou'].mean():.4f}, Std: {comparison_df['asb_iou'].std():.4f}, min: {comparison_df['asb_iou'].min():.4f} with idx {comparison_df['asb_iou'].idxmin()}, max: {comparison_df['asb_iou'].max():.4f} with idx {comparison_df['asb_iou'].idxmax()}")
print(f"Custom IoU - Mean: {comparison_df['custom_iou'].mean():.4f}, Std: {comparison_df['custom_iou'].std():.4f}, min: {comparison_df['custom_iou'].min():.4f} with idx {comparison_df['custom_iou'].idxmin()}, max: {comparison_df['custom_iou'].max():.4f}")

# Create comparison plots - adjusted for dynamic number of parameters
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# IoU comparison
axes[0, 0].hist(comparison_df['asb_iou'], bins=50, alpha=0.7, label='ASB IoU', color='blue')
axes[0, 0].hist(comparison_df['custom_iou'], bins=50, alpha=0.7, label='Custom IoU', color='red')
axes[0, 0].set_xlabel('IoU Value')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('IoU Distribution Comparison')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# IoU scatter plot
axes[0, 1].scatter(comparison_df['asb_iou'], comparison_df['custom_iou'], alpha=0.6, s=10)
axes[0, 1].plot([0, 1], [0, 1], 'r--', alpha=0.8)
axes[0, 1].set_xlabel('ASB IoU')
axes[0, 1].set_ylabel('Custom IoU')
axes[0, 1].set_title('IoU Correlation')
axes[0, 1].grid(True, alpha=0.3)

# Parameter comparison - upper weights (dynamic)
for i in range(dim_per_side):
    axes[1, 0].scatter(comparison_df[f'asb_upper_{i}'], comparison_df[f'custom_upper_{i}'], 
                      alpha=0.6, s=10, label=f'Upper {i}')
axes[1, 0].set_xlabel('ASB Parameters')
axes[1, 0].set_ylabel('Custom Parameters')
axes[1, 0].set_title('Upper Weight Parameters Comparison')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Parameter comparison - lower weights (dynamic)
for i in range(dim_per_side):
    axes[1, 1].scatter(comparison_df[f'asb_lower_{i}'], comparison_df[f'custom_lower_{i}'], 
                      alpha=0.6, s=10, label=f'Lower {i}')
axes[1, 1].set_xlabel('ASB Parameters')
axes[1, 1].set_ylabel('Custom Parameters')
axes[1, 1].set_title('Lower Weight Parameters Comparison')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
#get all indexes with iou under 0.8
iou_threshold = 0.9
underperforming_indexes = [i for i, iou in enumerate(comparison_df['asb_iou']) if iou < iou_threshold]
print(f"Indexes with IoU under {iou_threshold}: {underperforming_indexes}")

### Interim conclusion
The custom CST optimization does not consistently outperform the AeroSandbox Kulfan extraction in this benchmark.

For downstream dataset construction, the AeroSandbox parameterization is used as the default.

## Build CST + NeuralFoil dataframe
Next, create a single table that combines geometry descriptors (Kulfan/CST parameters) with aerodynamic outputs predicted by NeuralFoil.

### Aerodynamic evaluation settings
All NeuralFoil evaluations in this block use fixed conditions for comparability:
- Angle of attack: $\alpha = 5^\circ$
- Reynolds number: $Re = 10^6$

In [ ]:
# Initialize lists to store the parameters and aerodynamic data
asb_upper_weights_all = []
asb_lower_weights_all = []
asb_leading_edge_weights = []

# Aerodynamic data from NeuralFoil
analysis_confidence_all = []
CL_all = []
CD_all = []
CM_all = []
Cpmin_all = []
Top_Xtr_all = []
Bot_Xtr_all = []
mach_crit_all = []
cluster = []
 
# Settings for aerodynamic analysis
alpha_analysis = 5  # angle of attack in degrees
Re_analysis = 1e6    # Reynolds number
model_size = "xxxlarge"  # NeuralFoil model size

# Process each airfoil in the dataset
print(f"Processing {len(X_airfoils)} airfoils...")
start_time = time.time()

for i in range(len(X_airfoils)):
    if i % 50 == 0:  # Progress indicator
        print(f"Processing airfoil {i}/{len(X_airfoils)}")
    
    try:
        # Get ASB Kulfan parameters for current airfoil
        asb_results = get_kulfan_parameters(X_airfoils[i], 
                                           n_weights_per_side=dim_per_side, 
                                           use_leading_edge_modification=True)
        
        # Store ASB results
        asb_upper_weights_all.append(asb_results["upper_weights"])
        asb_lower_weights_all.append(asb_results["lower_weights"])
        asb_leading_edge_weights.append(asb_results["leading_edge_weight"])
        
        # Get aerodynamic polars using NeuralFoil
        af = Airfoil(name=f"Airfoil_{i}", coordinates=X_airfoils[i])
        polars = af.get_aero_from_neuralfoil(alpha=alpha_analysis, 
                                            Re=Re_analysis, 
                                            model_size=model_size)
        
        # Extract aerodynamic values
        analysis_confidence_all.append(polars.get('analysis_confidence', [0.0])[0])
        CL_all.append(polars.get('CL', [0.0])[0])
        CD_all.append(polars.get('CD', [0.0])[0])
        CM_all.append(polars.get('CM', [0.0])[0])
        Cpmin_all.append(polars.get('Cpmin', [0.0])[0])
        Top_Xtr_all.append(polars.get('Top_Xtr', [0.0])[0])
        Bot_Xtr_all.append(polars.get('Bot_Xtr', [0.0])[0])
        mach_crit_all.append(polars.get('mach_crit', [0.0])[0])
        cluster.append(0)

    except Exception as e:
        print(f"Processing failed for airfoil {i}: {e}")
        # Fill with default values if CST fitting fails
        asb_upper_weights_all.append([0.0] * dim_per_side)
        asb_lower_weights_all.append([0.0] * dim_per_side)
        asb_leading_edge_weights.append(0.0)
        
        analysis_confidence_all.append(0.0)
        CL_all.append(0.0)
        CD_all.append(0.0)
        CM_all.append(0.0)
        Cpmin_all.append(0.0)
        Top_Xtr_all.append(0.0)
        Bot_Xtr_all.append(0.0)
        mach_crit_all.append(0.0)
        cluster.append(0)

end_time = time.time()
print(f"Processing completed in {end_time - start_time:.2f} seconds")

# Create the comprehensive DataFrame
data = {}

#ASB CST parameters - dynamically create columns based on dim_per_side
for i in range(dim_per_side):
    data[f'asb_upper_{i}'] = [uw[i] if len(uw) > i else 0.0 for uw in asb_upper_weights_all]
    data[f'asb_lower_{i}'] = [lw[i] if len(lw) > i else 0.0 for lw in asb_lower_weights_all]

data['asb_leading_edge_weight'] = asb_leading_edge_weights

# Aerodynamic data from NeuralFoil
data['analysis_confidence'] = analysis_confidence_all
data['CL'] = CL_all
data['CL/CD'] = [cl/cd if cd != 0 else 0 for cl, cd in zip(CL_all, CD_all)]
data['CD'] = CD_all
data['CM'] = CM_all
data['Cpmin'] = Cpmin_all
data['Top_Xtr'] = Top_Xtr_all
data['Bot_Xtr'] = Bot_Xtr_all
data['mach_crit'] = mach_crit_all
data['cluster'] = cluster

# Create the DataFrame
airfoil_df = pd.DataFrame(data)

# Display the DataFrame info and statistics
print(f"\nDataFrame created with shape: {airfoil_df.shape}")
print(f"Number of CST dimensions per side: {dim_per_side}")
print(f"Analysis conditions: α={alpha_analysis}°, Re={Re_analysis:.0e}")

print("\nFirst 10 rows:")
print(airfoil_df.head(10))

print("\nDataFrame info:")
print(airfoil_df.info())

print("\nBasic statistics:")
print(airfoil_df.describe())


## Clustering quality baseline
Before applying clustering algorithms, compute average pairwise distances in geometry, aero, and performance subspaces.

This baseline helps compare whether later clustering methods increase intra-cluster compactness.

In [ ]:
# Baseline quality metrics before clustering (all samples currently in one group).
avg_cst, avg_aero, avg_perf = assess_cluster_quality_simple(
    airfoil_df, airfoil_df['cluster'], dim_per_side=dim_per_side
)

print("Average Within-Cluster Distances:")
print(f"  Avg CST distance: {avg_cst:.4f}")
print(f"  Avg Aero distance: {avg_aero:.4f}")
print(f"  Avg Perf distance: {avg_perf:.4f}")

In [ ]:
# Standardize the data before applying dimensionality reduction
if "cluster" in airfoil_df.columns:
    airfoil_df = airfoil_df.drop(columns=['cluster'])
    print("Dropped 'cluster' column for dimensionality reduction. --> else cluster id will be embedded")
scaler_aero = StandardScaler()
#only cluster with aerodynamic data
aero_scaled = scaler_aero.fit_transform(airfoil_df[['CL', 'CD', 'CM'] +
                                                     ['asb_leading_edge_weight'] +
                                                     [f'asb_upper_{i}' for i in range(dim_per_side)] +
                                                     [f'asb_lower_{i}' for i in range(dim_per_side)]])
# Apply PCA
print("Applying PCA...")
pca_aero = PCA(n_components=2)
pca_aero_result = pca_aero.fit_transform(aero_scaled)
print(f"PCA explained variance ratio: {pca_aero.explained_variance_ratio_}")
print(f"Total explained variance: {pca_aero.explained_variance_ratio_.sum():.3f}")

# Apply t-SNE
print("Applying t-SNE...")
tsne_aero = TSNE(n_components=2, random_state=42, perplexity=25, max_iter=1000)
tsne_aero_result = tsne_aero.fit_transform(aero_scaled)

print("All dimensionality reduction techniques completed for CST + Aerodynamic data!")

In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Row 1: Basic scatter plots
# PCA visualization
axes[0].scatter(pca_aero_result[:, 0], pca_aero_result[:, 1], alpha=0.6, s=10)
axes[0].set_title(f'PCA - CST + Aero (Explained Variance: {pca_aero.explained_variance_ratio_.sum():.3f})')
axes[0].set_xlabel('First Principal Component')
axes[0].set_ylabel('Second Principal Component')
axes[0].grid(True, alpha=0.3)

# t-SNE visualization
axes[1].scatter(tsne_aero_result[:, 0], tsne_aero_result[:, 1], alpha=0.6, s=10)
axes[1].set_title('t-SNE - CST + Aero')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Additional plot for L/D ratio
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Calculate L/D ratio (avoid division by zero)
cl_values = airfoil_df['CL'].values
cd_values = airfoil_df['CD'].values
ld_ratio = cl_values / np.where(cd_values == 0, 1e-6, cd_values)
#ld_ratio = airfoil_df['CL/CD'].values

# Plot L/D ratio
scatter7 = axes[0].scatter(pca_aero_result[:, 0], pca_aero_result[:, 1], 
                          c=ld_ratio, cmap='coolwarm', s=10, alpha=0.7)
axes[0].set_title('PCA - Colored by L/D Ratio')
axes[0].set_xlabel('First Principal Component')
axes[0].set_ylabel('Second Principal Component')
axes[0].grid(True, alpha=0.3)
plt.colorbar(scatter7, ax=axes[0], label='L/D')

scatter8 = axes[1].scatter(tsne_aero_result[:, 0], tsne_aero_result[:, 1], 
                          c=ld_ratio, cmap='coolwarm', s=10, alpha=0.7)
axes[1].set_title('t-SNE - Colored by L/D Ratio')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].grid(True, alpha=0.3)
plt.colorbar(scatter8, ax=axes[1], label='L/D')

plt.tight_layout()
plt.show()

In [ ]:
# GMM clustering with model selection
print("Applying GMM clustering with model selection...")

# Test different numbers of components
n_components_range = range(2, 50)
aic_scores = []
bic_scores = []
silhouette_scores_gmm = []

for n_components in n_components_range:
    # Fit GMM
    gmm = GaussianMixture(n_components=n_components, 
                         covariance_type='tied',  # Can be 'full', 'tied', 'diag', 'spherical'
                         random_state=42)
    gmm.fit(aero_scaled)
    
    # Get cluster assignments (hard clustering)
    cluster_labels = gmm.predict(aero_scaled)
    
    # Calculate scores
    aic_scores.append(gmm.aic(aero_scaled))
    bic_scores.append(gmm.bic(aero_scaled))
    
    # Calculate silhouette score
    if len(np.unique(cluster_labels)) > 1:  # Need at least 2 clusters for silhouette
        sil_score = silhouette_score(aero_scaled, cluster_labels)
        silhouette_scores_gmm.append(sil_score)
    else:
        silhouette_scores_gmm.append(-1)

# Find optimal number of components
optimal_n_aic = n_components_range[np.argmin(aic_scores)]
optimal_n_bic = n_components_range[np.argmin(bic_scores)]
optimal_n_sil = n_components_range[np.argmax(silhouette_scores_gmm)]

print(f"Optimal components - AIC: {optimal_n_aic}, BIC: {optimal_n_bic}, Silhouette: {optimal_n_sil}")

# Plot model selection results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(n_components_range, aic_scores, 'bo-')
axes[0].axvline(x=optimal_n_aic, color='r', linestyle='--', label=f'Optimal: {optimal_n_aic}')
axes[0].set_xlabel('Number of Components')
axes[0].set_ylabel('AIC Score')
axes[0].set_title('AIC vs Number of Components')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(n_components_range, bic_scores, 'go-')
axes[1].axvline(x=optimal_n_bic, color='r', linestyle='--', label=f'Optimal: {optimal_n_bic}')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('BIC Score')
axes[1].set_title('BIC vs Number of Components')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(n_components_range, silhouette_scores_gmm, 'mo-')
axes[2].axvline(x=optimal_n_sil, color='r', linestyle='--', label=f'Optimal: {optimal_n_sil}')
axes[2].set_xlabel('Number of Components')
axes[2].set_ylabel('Silhouette Score')
axes[2].set_title('Silhouette Score vs Number of Components')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Use BIC for final model selection (generally more conservative)
optimal_n_components = optimal_n_bic
print(f"Using {optimal_n_components} components based on BIC")

In [ ]:
# Fit final GMM model
print(f"Fitting final GMM model with {optimal_n_components} components...")

gmm_final = GaussianMixture(n_components=optimal_n_components, 
                           covariance_type='full',
                           random_state=42)
gmm_final.fit(aero_scaled)

# Get hard cluster assignments
gmm_cluster_labels = gmm_final.predict(aero_scaled)

# Get soft cluster assignments (probabilities)
gmm_probabilities = gmm_final.predict_proba(aero_scaled)

# Add GMM cluster labels to your dataframe
airfoil_df['cluster'] = gmm_cluster_labels

print(f"GMM clustering completed!")
print(f"Assigned {len(np.unique(gmm_cluster_labels))} GMM clusters")
print(f"Cluster distribution: {np.bincount(gmm_cluster_labels)}")

# Visualize GMM clusters in t-SNE space
plt.figure(figsize=(15, 6))

# Plot 1: Hard clustering
plt.subplot(1, 2, 1)
scatter_gmm = plt.scatter(tsne_aero_result[:, 0], tsne_aero_result[:, 1], 
                         c=gmm_cluster_labels, cmap='viridis', s=20, alpha=0.7)
plt.title(f't-SNE - GMM Hard Clustering ({optimal_n_components} components)')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.colorbar(scatter_gmm, label='GMM Cluster ID')
plt.grid(True, alpha=0.3)

# Plot 2: Uncertainty visualization (entropy of probabilities)
plt.subplot(1, 2, 2)
# Calculate entropy (uncertainty) for each point
entropy = -np.sum(gmm_probabilities * np.log(gmm_probabilities + 1e-10), axis=1)
scatter_entropy = plt.scatter(tsne_aero_result[:, 0], tsne_aero_result[:, 1], 
                             c=entropy, cmap='viridis', s=20, alpha=0.7)
plt.title('t-SNE - GMM Clustering Uncertainty')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.colorbar(scatter_entropy, label='Clustering Uncertainty (Entropy)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## K-means clustering on reduced feature space
This section tunes $k$ with silhouette score, fits K-means in standardized feature space, and visualizes labels on t-SNE coordinates.

In [ ]:
# Usage
avg_cst, avg_aero, avg_perf = assess_cluster_quality_simple(
    airfoil_df, airfoil_df['cluster'], dim_per_side=dim_per_side
)

print("Average Within-Cluster Distances:")
print(f"  Avg CST distance: {avg_cst:.4f}")
print(f"  Avg Aero distance: {avg_aero:.4f}")
print(f"  Avg Perf distance: {avg_perf:.4f}")

In [ ]:
# Use your scaled 15D data (aero_scaled from your existing code)
# Determine optimal number of clusters using elbow method or silhouette score
silhouette_scores = []
k_range = range(10, 30)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    cluster_labels = kmeans.fit_predict(aero_scaled)
    silhouette_avg = silhouette_score(aero_scaled, cluster_labels)
    silhouette_scores.append(silhouette_avg)

# Find optimal k
optimal_k = k_range[np.argmax(silhouette_scores)]

# Perform clustering in 15D space
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42)
cluster_labels = kmeans_final.fit_predict(aero_scaled)

# Add cluster labels to your dataframe
airfoil_df['cluster'] = cluster_labels

plt.figure(figsize=(10, 6))
# Plot clusters in t-SNE space
scatter_clusters = plt.scatter(tsne_aero_result[:, 0], tsne_aero_result[:, 1], 
                                c=cluster_labels, cmap='viridis', s=10, alpha=0.7)
plt.title(f't-SNE - Colored by Clusters (k={optimal_k})')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.grid(True, alpha=0.3)
plt.colorbar(scatter_clusters, label='Cluster', ticks=range(optimal_k))

plt.tight_layout()
plt.show()

In [ ]:
# Usage
avg_cst, avg_aero, avg_perf = assess_cluster_quality_simple(
    airfoil_df, airfoil_df['cluster'], dim_per_side=dim_per_side
)

print("Average Within-Cluster Distances:")
print(f"  Avg CST distance: {avg_cst:.4f}")
print(f"  Avg Aero distance: {avg_aero:.4f}")
print(f"  Avg Perf distance: {avg_perf:.4f}")

In [ ]:
# Create hexbin plot and extract bin assignments
plt.figure(figsize=(12, 10))
hb = plt.hexbin(tsne_aero_result[:, 0], tsne_aero_result[:, 1], 
                gridsize=6, cmap='viridis', mincnt=15, alpha=0.7)

# Extract hexagonal bin information
x_coords = tsne_aero_result[:, 0]
y_coords = tsne_aero_result[:, 1]

# Get hexagon centers from the hexbin plot
hex_centers = hb.get_offsets()

# Assign each point to a hexagonal cluster
hex_cluster_labels = assign_hex_clusters(x_coords, y_coords, hex_centers)

# Add hexagonal cluster labels to your dataframe (replacing the old cluster column)
airfoil_df['cluster'] = hex_cluster_labels

plt.title('t-SNE Hexagonal Representation', fontsize=16)
plt.scatter(tsne_aero_result[:, 0], tsne_aero_result[:, 1], s=20, alpha=0.5)
plt.xlabel('t-SNE 1', fontsize=14)
plt.ylabel('t-SNE 2', fontsize=14)
plt.colorbar(hb, label='Point Density')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Assigned {len(np.unique(hex_cluster_labels))} hexagonal clusters")
print(f"Cluster distribution: {np.bincount(hex_cluster_labels)}")

# Visualize the hexagonal clusters
plt.figure(figsize=(12, 10))
scatter = plt.scatter(tsne_aero_result[:, 0], tsne_aero_result[:, 1], 
                     c=hex_cluster_labels, cmap='viridis', s=20, alpha=0.7)
plt.title('t-SNE - Colored by Hexagonal Clusters', fontsize=16)
plt.xlabel('t-SNE 1', fontsize=14)
plt.ylabel('t-SNE 2', fontsize=14)
plt.colorbar(scatter, label='Hexagonal Cluster ID')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Usage
avg_cst, avg_aero, avg_perf = assess_cluster_quality_simple(
    airfoil_df, airfoil_df['cluster'], dim_per_side=dim_per_side
)

print("Average Within-Cluster Distances:")
print(f"  Avg CST distance: {avg_cst:.4f}")
print(f"  Avg Aero distance: {avg_aero:.4f}")
print(f"  Avg Perf distance: {avg_perf:.4f}")

In [ ]:
# SOM clustering on original dimensional data
print("Applying SOM clustering...")

# Parameters for SOM
som_x, som_y = 5, 5  # 5x5 grid = 25 clusters
som_input_len = aero_scaled.shape[1]  # Number of features
som_sigma = 0.5  # Neighborhood function parameter
som_learning_rate = 0.1  # Learning rate
som_num_iterations = 5000  # Number of training iterations

# Initialize and train SOM
som = MiniSom(som_x, som_y, som_input_len, 
              sigma=som_sigma, 
              learning_rate=som_learning_rate,
              random_seed=42)

# Initialize weights
som.random_weights_init(aero_scaled)

# Train the SOM
print(f"Training SOM with {som_num_iterations} iterations...")
som.train(aero_scaled, som_num_iterations, verbose=True)

# Get cluster assignments for each data point
som_cluster_labels = []

for i, sample in enumerate(aero_scaled):
    # Find the winning neuron (BMU - Best Matching Unit)
    winning_neuron = som.winner(sample)
    # Convert 2D coordinates to 1D cluster ID
    cluster_id = winning_neuron[0] * som_y + winning_neuron[1]
    som_cluster_labels.append(cluster_id)

som_cluster_labels = np.array(som_cluster_labels)

# Add SOM cluster labels to your dataframe
airfoil_df['cluster'] = som_cluster_labels

print(f"SOM clustering completed!")
print(f"Assigned {len(np.unique(som_cluster_labels))} SOM clusters")
print(f"Cluster distribution: {np.bincount(som_cluster_labels)}")

# Visualize SOM clusters in t-SNE space
plt.figure(figsize=(15, 6))

# Plot 1: SOM clusters in t-SNE space
plt.subplot(1, 2, 1)
scatter_som = plt.scatter(tsne_aero_result[:, 0], tsne_aero_result[:, 1], 
                         c=som_cluster_labels, cmap='viridis', s=20, alpha=0.7)
plt.title('t-SNE - Colored by SOM Clusters (25 clusters)', fontsize=14)
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.colorbar(scatter_som, label='SOM Cluster ID')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Usage
avg_cst, avg_aero, avg_perf = assess_cluster_quality_simple(
    airfoil_df, airfoil_df['cluster'], dim_per_side=dim_per_side
)

print("Average Within-Cluster Distances:")
print(f"  Avg CST distance: {avg_cst:.4f}")
print(f"  Avg Aero distance: {avg_aero:.4f}")
print(f"  Avg Perf distance: {avg_perf:.4f}")

## Save processed dataset
Export the current dataframe (including selected cluster labels) to the processed data folder for reuse in model training and analysis notebooks.

In [ ]:
# Save dtaframe to CSV
print(os.getcwd())
airfoil_df.to_csv("../../data/processed/airfoil_with_polars.csv", index=False)

## VAE-based clustering workflow
This section trains a preprocessing VAE and uses its latent representation as an alternative clustering space to compare against classical methods.

In [ ]:
# Path-agnostic project root detection (search upward for pyproject.toml).
cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").exists()), cwd)
os.chdir(project_root)
print(f"Using project root: {project_root}")

# Remove the module from cache --> enforce re-import
if 'models.preprocessing_VAE_data_module' in sys.modules:
    del sys.modules['models.preprocessing_VAE_data_module']
if 'models.preprocessing_VAE' in sys.modules:
    del sys.modules['models.preprocessing_VAE']

np.random.seed(42)
torch.manual_seed(42)
random.seed(42)


In [ ]:
# Path-agnostic MLflow setup for preprocessing VAE runs.
mlflow_tracking_dir = project_root / "outputs" / "preprocessing_VAE_training_runs"
mlflow_tracking_dir.mkdir(parents=True, exist_ok=True)
mlflow_tracking_uri = mlflow_tracking_dir.resolve().as_uri()

print(f"MLflow tracking URI: {mlflow_tracking_uri}")
print("To launch MLflow UI from any terminal:")
print(f'mlflow ui --backend-store-uri "{mlflow_tracking_uri}" --port 8080')

model, trainer, cfg, data_module = train_preprocessing_vae_with_hydra(
    tracking_uri=mlflow_tracking_uri,
)


In [ ]:
# Training stability analysis.
num_runs = 5
seeds = random.sample(range(1, 10000), num_runs)
for seed in seeds:
    model, trainer, cfg, data_module = train_preprocessing_vae_with_hydra(
        seed=seed,
        save_curves=True,
        tracking_uri=mlflow_tracking_uri,
    )
    del data_module
    del model

    gc.collect()
    torch.cuda.empty_cache()


### Latent-space diagnostics
The following visualization utilities map samples into latent space and inspect relationships with aerodynamic/geometry attributes.

In [ ]:
# Main execution with robust model recovery logic
model_in_memory = "model" in locals() and model is not None

if not model_in_memory:
    checkpoint_path = project_root / "models" / "prep_VAE" / "preprocessing_VAE_v1p1.pt"
    if checkpoint_path.exists():
        print(f"No in-memory model found. Loading checkpoint from: {checkpoint_path}")
        model, model_details = VAE.load(checkpoint_path, add_details=True)
        model = model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
        model.eval()
    else:
        print("No in-memory model and no checkpoint found.")
        print("Run the VAE training cell first, or place a checkpoint at models/prep_VAE/preprocessing_VAE_v1p1.pt")

if "model" in locals() and model is not None:
    print("Mapping dataset to VAE latent space...")
    data_path = "data/processed/airfoil_with_polars.csv"
    df = pd.read_csv(data_path)
    
    df_for_latent = df.copy()
    df_features = df.copy()
    if "cluster" in df_features.columns:
        df_features = df_features.drop(columns=['cluster'])
    
    feature_columns = ['asb_leading_edge_weight'] + \
                     [f'asb_upper_{i}' for i in range(3)] + \
                     [f'asb_lower_{i}' for i in range(3)]
    
    if model.scaler_mean is not None and model.scaler_scale is not None:
        print("Using VAE's built-in scaler parameters")
        scaler = StandardScaler()
        scaler.mean_ = model.scaler_mean.cpu().numpy()
        scaler.scale_ = model.scaler_scale.cpu().numpy()
        scaler.n_features_in_ = len(feature_columns)
    else:
        print("Creating new scaler")
        scaler = StandardScaler()
        scaler.fit(df_features[feature_columns].values.astype('float32'))
    
    latent_means, latent_logvars, scaled_data = map_dataset_to_latent_space(
        model, scaler, df_features, feature_columns
    )
    
    distance_outlier_mask, variance_outlier_mask = analyze_latent_distributions_l2(latent_means, latent_logvars)

    visualize_latent_space(latent_means, df_for_latent, "VAE 2D Latent Space")
    
    grid_points, decoded_grid, xx, yy = sample_from_latent_grid(
        model, scaler, feature_columns, grid_size=5
    )
    
    visualize_latent_grid_sampling(grid_points, decoded_grid, xx, yy, feature_columns, grid_size=5)
    
    print(f"\nLatent Space Statistics:")
    print(f"  Latent Dim 1 - Mean: {latent_means[:, 0].mean():.3f}, Std: {latent_means[:, 0].std():.3f}")
    print(f"  Latent Dim 2 - Mean: {latent_means[:, 1].mean():.3f}, Std: {latent_means[:, 1].std():.3f}")
    print(f"  Latent space coverage: [{latent_means.min():.2f}, {latent_means.max():.2f}]")
else:
    print("No trained VAE model available. Please run the VAE training cell first.")

In [ ]:
print(OmegaConf.to_yaml(cfg))
#model.save("models/prep_VAE/preprocessing_VAE_v1p1.pt", full_config=cfg)

In [ ]:
visualize_outlier_airfoils(df, distance_outlier_mask, latent_means)

remove_distance_outliers = True
remove_variance_outliers = False

if remove_distance_outliers:
    outlier_indices = df[distance_outlier_mask].index.tolist()
    df = df[~distance_outlier_mask].reset_index(drop=True)
    latent_means = latent_means[~distance_outlier_mask]
    latent_logvars = latent_logvars[~distance_outlier_mask]
    print(f"Dataframe size after removing distance outliers: {len(latent_means)}")
    print(f"Removed Airfoils with IDs: {outlier_indices}")
if remove_variance_outliers:
    outlier_indices = df[variance_outlier_mask].index.tolist()
    df = df[~variance_outlier_mask].reset_index(drop=True)
    latent_means = latent_means[~variance_outlier_mask]
    latent_logvars = latent_logvars[~variance_outlier_mask]
    print(f"Removed Airfoils with IDs: {outlier_indices}")

### Outlier candidate selection in latent space
This section identifies latent points far from the centroid and with high uncertainty, then maps them back to airfoil IDs for inspection.

In [ ]:
rmse, latent_coord = visualize_airfoil_reconstruction(
    model, scaler, df.iloc[505], title="Sample Airfoil Reconstruction"
 )
print(f"Reconstruction RMSE: {rmse:.6f}")
print(f"Latent coordinate: {latent_coord}")

In [ ]:


k_range = range(5, 20)
silhouette_scores = []
calinski_harabasz_scores = []
davies_bouldin_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    cluster_labels = kmeans.fit_predict(latent_means)

    silhouette_scores.append(silhouette_score(latent_means, cluster_labels))
    davies_bouldin_scores.append(davies_bouldin_score(latent_means, cluster_labels))

# Find optimal k for each metric
optimal_k_silhouette = k_range[np.argmax(silhouette_scores)]      
optimal_k_davies = k_range[np.argmin(davies_bouldin_scores)]      

print(f"Optimal k (Silhouette): {optimal_k_silhouette}, Value: {max(silhouette_scores):.4f}")
print(f"Optimal k (Davies-Bouldin): {optimal_k_davies}, Value: {min(davies_bouldin_scores):.4f}")


fig, axes = plt.subplots(1, 2, figsize=(18, 5))

axes[0].plot(k_range, silhouette_scores, 'bo-')
axes[0].axvline(optimal_k_silhouette, color='r', linestyle='--', label=f'Optimal: {optimal_k_silhouette}')
axes[0].set_xlabel('Number of Clusters')

n_components_range = range(2, 50)
axes[0].set_ylabel('Silhouette Score')
axes[0].set_title('Silhouette Score vs k')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_range, davies_bouldin_scores, 'mo-')
axes[1].axvline(optimal_k_davies, color='r', linestyle='--', label=f'Optimal: {optimal_k_davies}')
axes[1].set_xlabel('Number of Clusters')
axes[1].set_ylabel('Davies-Bouldin Score')
axes[1].set_title('Davies-Bouldin Score vs k')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

n_clusters = 14# Choose based on metrics

# Fit final KMeans with optimal k (e.g., silhouette)
kmeans_final = KMeans(n_clusters=n_clusters, random_state=42)
final_labels = kmeans_final.fit(latent_means).labels_

plt.figure(figsize=(10, 6))
plt.scatter(latent_means[:, 0], latent_means[:, 1], c=final_labels, cmap='viridis', s=10, alpha=0.7)
plt.title('Final Clustering Results')
plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')
plt.colorbar(label='Cluster')
plt.show()

df['cluster'] = final_labels

In [ ]:
n_components_range = range(5, 20)
aic_scores = []
bic_scores = []
silhouette_scores_gmm = []
davies_bouldin_scores_gmm = []
typ = 'full'  # Can be 'full', 'tied', 'diag', 'spherical'

for n_components in n_components_range:
    gmm = GaussianMixture(n_components=n_components, 
                         covariance_type=typ,
                         random_state=42)
    gmm.fit(latent_means)
    
    cluster_labels = gmm.predict(latent_means)

    aic_scores.append(gmm.aic(latent_means))
    bic_scores.append(gmm.bic(latent_means))
    silhouette_scores_gmm.append(silhouette_score(latent_means, cluster_labels))
    davies_bouldin_scores_gmm.append(davies_bouldin_score(latent_means, cluster_labels))

optimal_n_aic = n_components_range[np.argmin(aic_scores)]
optimal_n_bic = n_components_range[np.argmin(bic_scores)]
optimal_n_sil = n_components_range[np.argmax(silhouette_scores_gmm)]
optimal_n_db = n_components_range[np.argmin(davies_bouldin_scores_gmm)]

print(f"Optimal components - AIC: {optimal_n_aic}, BIC: {optimal_n_bic}")
print(f"Optimal components - Silhouette: {optimal_n_sil}, Davies-Bouldin: {optimal_n_db}")

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

axes[0, 0].plot(n_components_range, aic_scores, 'bo-')
axes[0, 0].axvline(x=optimal_n_aic, color='r', linestyle='--', label=f'Optimal: {optimal_n_aic}')
axes[0, 0].set_xlabel('Number of Components')
axes[0, 0].set_ylabel('AIC Score')
axes[0, 0].set_title('AIC vs Number of Components')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0,1].plot(n_components_range, bic_scores, 'go-')
axes[0,1].axvline(x=optimal_n_bic, color='r', linestyle='--', label=f'Optimal: {optimal_n_bic}')
axes[0,1].set_xlabel('Number of Components')
axes[0,1].set_ylabel('BIC Score')
axes[0,1].set_title('BIC vs Number of Components')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(n_components_range, silhouette_scores_gmm, 'mo-')
axes[1,0].axvline(x=optimal_n_sil, color='r', linestyle='--', label=f'Optimal: {optimal_n_sil}')
axes[1,0].set_xlabel('Number of Components')
axes[1,0].set_ylabel('Silhouette Score')
axes[1,0].set_title('Silhouette Score vs Number of Components')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(n_components_range, davies_bouldin_scores_gmm, 'co-')
axes[1,1].axvline(x=optimal_n_db, color='r', linestyle='--', label=f'Optimal: {optimal_n_db}')
axes[1,1].set_xlabel('Number of Components')
axes[1,1].set_ylabel('Davies-Bouldin Score')
axes[1,1].set_title('Davies-Bouldin Score vs Number of Components')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

n_clusters = 12
gmm = GaussianMixture(n_components=n_clusters,
                         covariance_type=typ,
                         random_state=42)
gmm.fit(latent_means)
gmm_cluster_labels = gmm.predict(latent_means)
gmm_probabilities = gmm.predict_proba(latent_means)

plt.figure(figsize=(10, 6))
plt.scatter(latent_means[:, 0], latent_means[:, 1], c=gmm_cluster_labels, cmap='viridis', s=10, alpha=0.7)
plt.title('Final Clustering Results')
plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')
plt.colorbar(label='Cluster')
plt.show()

print(f"Scores: Shillouette: {silhouette_score(latent_means, gmm_cluster_labels):.4f}, Davies-Bouldin: {davies_bouldin_score(latent_means, gmm_cluster_labels):.4f}")

df['cluster'] = gmm_cluster_labels

gmm_p_x_given_k = compute_p_x_given_k(gmm, latent_means)

for i in range(gmm_p_x_given_k.shape[1]):
    print(f"Cluster {i} summed probability: {gmm_p_x_given_k[:, i].sum():.4f}")
    df[f'gmm_prob_cluster_{i}'] = gmm_p_x_given_k[:, i]/gmm_p_x_given_k[:, i].sum()
    print(f"Cluster {i} summed probability: {df[f'gmm_prob_cluster_{i}'].sum():.4f}")
print((df["cluster"]==df[[f"gmm_prob_cluster_{i}" for i in range(gmm_p_x_given_k.shape[1])]].idxmax(axis=1).str.extract(r"(\d+)$")[0].astype(int)).all() == True)

print("Mismatches indices:", df.index[df["cluster"].values != df[[f'gmm_prob_cluster_{i}' for i in range(gmm_p_x_given_k.shape[1])]].values.argmax(axis=1)].tolist())

## Explore selected outliers
Visualize and compare representative outlier airfoils to understand which geometric patterns are hard for the latent model.

In [ ]:
cluster_id = 12  # Example cluster ID
airfoils_in_cluster = df[df['cluster'] == cluster_id]
dim_per_side = 3  # Number of CST parameters per side
n_points_per_side = 96  # Number of points per side for airfoil coordinates

# Get feature columns (exclude the cluster column)
feature_columns = [col for col in df.columns if col != 'cluster']
cluster_features = airfoils_in_cluster[feature_columns]

print(f"Cluster {cluster_id} contains {len(airfoils_in_cluster)} airfoils")

# Sample 4 random airfoils from the cluster
n_samples = min(4, len(airfoils_in_cluster))
random_indices = np.random.choice(airfoils_in_cluster.index, size=n_samples, replace=False)

print(f"Selected airfoil indices: {random_indices}")

# Get the original features for the random airfoils (before scaling)
random_airfoils_original = df.loc[random_indices, feature_columns].values

# Create a 2x2 subplot for visualizing the 4 airfoils
fig, axes = plt.subplots(2, 2, figsize=(20, 10))
axes = axes.flatten()

# Create a mapping from feature names to indices
feature_name_to_idx = {name: idx for idx, name in enumerate(feature_columns)}

for i, (idx, original_features) in enumerate(zip(random_indices, random_airfoils_original)):
    # Extract CST parameters using correct indexing
    uw = []
    lw = []
    
    # Get upper weights
    for j in range(dim_per_side):
        upper_col = f'asb_upper_{j}'
        uw.append(original_features[feature_name_to_idx[upper_col]])
    
    # Get lower weights  
    for j in range(dim_per_side):
        lower_col = f'asb_lower_{j}'
        lw.append(original_features[feature_name_to_idx[lower_col]])
    
    # Get leading edge weight
    le = original_features[feature_name_to_idx['asb_leading_edge_weight']]
    
    # Create airfoil coordinates
    coords = get_kulfan_coordinates(
        lower_weights=lw, 
        upper_weights=uw, 
        leading_edge_weight=le, 
        n_points_per_side=n_points_per_side,
        TE_thickness=0
    )
    
    print(f"Airfoil {idx} - Upper weights: {uw}, Lower weights: {lw}, LE weight: {le}")

    # Plot the airfoil
    axes[i].axis('equal')
    axes[i].grid(True, alpha=0.3)
    axes[i].scatter(coords[:, 0], coords[:, 1], alpha=0.5, s=10, color='red')
    axes[i].set_title(f'Airfoil {idx} (Cluster {cluster_id})', fontsize=12)
    axes[i].set_xlabel('x/c')
    axes[i].set_ylabel('y/c')
    
    # Add aerodynamic info as text
    cl = original_features[feature_name_to_idx['CL']]
    cd = original_features[feature_name_to_idx['CD']]
    ld_ratio = cl / cd if cd != 0 else 0
    
    axes[i].text(0.02, 0.02, f'L/D: {ld_ratio:.1f}\nCL: {cl:.3f}\nCD: {cd:.4f}', 
                transform=axes[i].transAxes, fontsize=10, 
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

plt.tight_layout()
plt.suptitle(f'Random Airfoils from Cluster {cluster_id}', fontsize=16, y=1.02)
plt.show()

In [ ]:
#df.to_csv(r"data/processed/airfoil_with_polars_without_outliers_v2.csv", index=False)
#df.head()

## Cluster latent embeddings
Apply clustering directly in latent space and compare compactness/structure with earlier feature-space clustering results.

In [ ]:
#df = pd.read_csv("data/processed/airfoil_with_polars.csv")
#df.head()

df_for_latent = df.copy()

# Remove cluster column for the feature extraction but keep it for visualization
df_features = df.copy()
if "cluster" in df_features.columns:
    df_features = df_features.drop(columns=['cluster'])

feature_columns = ['asb_leading_edge_weight'] + \
                    [f'asb_upper_{i}' for i in range(3)] + \
                    [f'asb_lower_{i}' for i in range(3)]

# Map to latent space
latent_means, latent_logvars, scaled_data = map_dataset_to_latent_space(
    model, scaler, df_features, feature_columns
)

gmm = GaussianMixture(n_components=n_clusters,
                         covariance_type=typ,  # Can be 'full', 'tied', 'diag', 'spherical'
                         random_state=42)
gmm.fit(latent_means)

gmm_cluster_labels = gmm.predict(latent_means)
gmm_probabilities = gmm.predict_proba(latent_means)
gmm_means = gmm.means_

if gmm.covariance_type == 'tied':
    covariances = [gmm.covariances_] * gmm.n_components
elif gmm.covariance_type == 'full':
    covariances = gmm.covariances_
elif gmm.covariance_type == 'diag':
    covariances = [np.diag(cov) for cov in gmm.covariances_]
elif gmm.covariance_type == 'spherical':
    covariances = [np.eye(2) * cov for cov in gmm.covariances_]

cmap = cm.get_cmap('tab20', gmm.n_components)
norm = plt.Normalize(vmin=0, vmax=gmm.n_components - 1)

ellipse_info = []
for i in range(gmm.n_components):
    mean = gmm_means[i]
    covariance = covariances[i]
    ellipse_info.append((mean, covariance, cmap(norm(i))))

cluster_rgba = [to_rgba(cmap(norm(i))) for i in range(gmm.n_components)]

blended_colors = []
for probs in gmm_probabilities:
    c = np.zeros(4)
    for i, p in enumerate(probs):
        c += p * np.array(cluster_rgba[i])
    c = np.clip(c, 0, 1)
    blended_colors.append(c)

plt.figure(figsize=(10, 6))
plt.scatter(gmm_means[:, 0], gmm_means[:, 1],
            s=45, c=[cmap(norm(i)) for i in range(gmm.n_components)],
            edgecolor='black', linewidth=0.6, zorder=5)
plt.scatter(latent_means[:, 0], latent_means[:, 1], c=blended_colors, s=10, alpha=0.7)
plt.title('Final Clustering Results')
plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=plt.gca(),ticks=np.arange(gmm.n_components))
cbar.set_label("Cluster")

for i, (mean, cov, color) in enumerate(ellipse_info):
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    angle = np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0])
    width, height = 2 * np.sqrt(eigenvalues) * 2
    ellipse = Ellipse(
        xy=mean, width=width, height=height, angle=np.degrees(angle),
        edgecolor=color, fc='None',
        lw=1.6, ls='--', alpha=0.85
    )
    plt.gca().add_patch(ellipse)

proxy_ellipse = Ellipse((0, 0), width=1, height=0.5, edgecolor='black', fc='None', lw=1, ls='--')
plt.legend([proxy_ellipse], ['2σ range (GMM cluster)'], loc='upper right', frameon=True)
plt.show()

### Filter outliers from final dataset
Remove selected outlier samples and create a cleaned dataframe for downstream training/evaluation.

In [ ]:
fig, axes = plt.subplots(gmm.n_components//2, 2, figsize=(18, 12))
axes = axes.flatten()
legend_handles = []
for i in range(gmm.n_components):
    mean = gmm_means[i]
    mean_tensor = torch.tensor(mean, dtype=torch.float32).unsqueeze(0).to(model.device)
    
    with torch.no_grad():
        decoded = model.decode(mean_tensor)
        decoded_np = decoded.cpu().numpy()
        decoded_original = scaler.inverse_transform(decoded_np)[0]

    coords_reconstructed = get_kulfan_coordinates(lower_weights=decoded_original[4:7],
                                              upper_weights=decoded_original[0:4],
                                              leading_edge_weight=decoded_original[0],
                                              TE_thickness=0,
                                              n_points_per_side=96)
    

    distances = np.linalg.norm(latent_means - gmm_means[i], axis=1)
    idx = np.argmin(distances)

    # Get CST parameters from df
    real_airfoil = df.iloc[idx]
    uw = [real_airfoil[f'asb_upper_{j}'] for j in range(3)]
    lw = [real_airfoil[f'asb_lower_{j}'] for j in range(3)]
    le = real_airfoil['asb_leading_edge_weight']

    # Original airfoil
    coords_original = get_kulfan_coordinates(
        lower_weights=lw,
        upper_weights=uw,
        leading_edge_weight=le,
        TE_thickness=0,
        n_points_per_side=96
    )

    axes[i].axis('equal')
    axes[i].grid(True, alpha=0.3)
    # Plot the lines
    line1 = axes[i].plot(coords_reconstructed[:, 0], coords_reconstructed[:, 1], 
                        alpha=1, color=ellipse_info[i][2], lw=2)[0]
    line2 = axes[i].plot(coords_original[:, 0], coords_original[:, 1], 
                        color=ellipse_info[i][2], alpha=0.7, lw=1, linestyle='--')[0]
    
    # Create legend handles only once (from first iteration) with correct labels
    if i == 0:
        # Create dummy lines with black color for consistent legend appearance
        line_solid = axes[i].plot([], [], color='black', lw=2, label='Reconstruction of GMM Mean')[0]
        line_dotted = axes[i].plot([], [], color='black', lw=2, linestyle='--', label='Closest embedded Real Airfoil')[0]
        legend_handles = [line_solid, line_dotted]
    axes[i].set_title(f'Airfoil of Cluster {i}, Latent_Mean {mean.round(2)}', fontsize=12)
    axes[i].set_xlabel('x/c')
    axes[i].set_ylabel('y/c')
        
fig.suptitle(f'Airfoil Reconstructions from GMM Cluster Means \n and Closest Embedded Real Airfoils', fontsize=16)
fig.legend(legend_handles, ['Reconstruction of GMM Mean', 'Closest embedded Real Airfoil'], 
          loc='upper right', bbox_to_anchor=(0.95, 1), ncol=1, fontsize=12)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
#plt.savefig("references/vae_gmm_cluster_airfoils_v2.svg", format="svg")
plt.show()
    


## Persist cleaned dataset
Save the filtered dataframe to disk so subsequent models can train on the outlier-cleaned airfoil set.

In [ ]:


fig, axes = plt.subplots(gmm.n_components//2, 2, figsize=(20, 12))
axes = axes.flatten()
legend_handles = []

for i in range(gmm.n_components):
    # Find the sample whose latent mean is closest to the cluster mean
    distances = np.linalg.norm(latent_means - gmm_means[i], axis=1)
    idx = np.argmin(distances)
    print(f"Cluster {i}: Closest latent mean index in dataset: {idx}")

    # Get CST parameters from df
    real_airfoil = df.iloc[idx]
    uw = [real_airfoil[f'asb_upper_{j}'] for j in range(3)]
    lw = [real_airfoil[f'asb_lower_{j}'] for j in range(3)]
    le = real_airfoil['asb_leading_edge_weight']

    # Original airfoil
    coords_original = get_kulfan_coordinates(
        lower_weights=lw,
        upper_weights=uw,
        leading_edge_weight=le,
        TE_thickness=0,
        n_points_per_side=96
    )

    # Pass through VAE: scale, encode, decode, inverse scale
    features = np.array([le] + uw + lw).reshape(1, -1).astype('float32')
    features_scaled = scaler.transform(features)
    features_tensor = torch.tensor(features_scaled).to(model.device)
    with torch.no_grad():
        latent, _ = model.encode(features_tensor)
        reconstructed = model.decode(latent)
        reconstructed_np = reconstructed.cpu().numpy()
        reconstructed_original = scaler.inverse_transform(reconstructed_np)[0]

    uw_rec = reconstructed_original[1:4]
    lw_rec = reconstructed_original[4:7]
    le_rec = reconstructed_original[0]

    coords_reconstructed = get_kulfan_coordinates(
        lower_weights=lw_rec,
        upper_weights=uw_rec,
        leading_edge_weight=le_rec,
        TE_thickness=0,
        n_points_per_side=96
    )

    axes[i].axis('equal')
    axes[i].grid(True, alpha=0.3)

    # Plot lines and store handles from the first iteration for legend
    line1 = axes[i].plot(coords_original[:, 0], coords_original[:, 1], 
                        color='black', lw=2, label='Original Airfoil')[0]
    axes[i].plot(coords_reconstructed[:, 0], coords_reconstructed[:, 1], 
                        color=ellipse_info[i][2], lw=2, linestyle='--', 
                        label='VAE Reconstruction')[0]
    
    # Store legend handles from first iteration
    if i == 0:
        line_dotted = axes[i].plot([], [], color='black', lw=2, linestyle='--', label='VAE Reconstruction')[0]
        legend_handles = [line1, line_dotted]

    axes[i].set_title(f'Cluster {i}, Airfoil {idx}', fontsize=12)
    axes[i].set_xlabel('x/c')
    axes[i].set_ylabel('y/c')

fig.suptitle('Original Airfoils vs VAE Reconstructions \n for Closest Airfoil Latent Means to Cluster Centers', fontsize=16)

fig.legend(legend_handles, ['Original Airfoil', 'VAE Reconstruction'], 
          loc='upper right', bbox_to_anchor=(0.95, 1), ncol=1, fontsize=12)
plt.tight_layout()
plt.subplots_adjust(top=0.92)  # Make room for the legend
#plt.savefig("references/vae_gmm_cluster_airfoils_reconstruction_v2.svg", format="svg")
plt.show()